In [0]:
# Load the cleaned movie titles and our AI recommendations
movies = spark.table("silver_movies")
recommendations = spark.table("gold_recommendations")

print("✨ Production Data Loaded. Ready for Dashboarding!")

In [0]:
from pyspark.sql.functions import col

# 1. LOAD DATA DIRECTLY FROM VOLUMES (The safest way in Serverless)
# This ensures we don't depend on "silver_movies" or "gold_recommendations" tables
try:
    # Load Movies for Titles
    movie_titles = spark.read.option("delimiter", "::").csv("/Volumes/workspace/default/raw_data/movies.dat") \
        .toDF("movieId", "title", "genres")
    
    # Load the Gold Recommendations Table we saved earlier
    gold_recs = spark.table("gold_recommendations")
except Exception as e:
    print(f"Checking data sources... Note: {e}")
    # Final backup: If the table was dropped, we read the parquet file we saved in 04
    gold_recs = spark.read.parquet("/Volumes/workspace/default/raw_data/temp_gold_recs")

def show_recommendations(target_user):
    # Filter for the specific user
    user_data = gold_recs.filter(col("userId") == target_user)
    
    # Join on the movie_1 column from your bypass logic
    # Note: Using .cast("string") ensures the IDs match perfectly during the join
    res = user_data.join(movie_titles, user_data.movie_1.cast("string") == movie_titles.movieId.cast("string")) \
        .select(
            col("userId"),
            col("title").alias("Top_Recommendation"),
            col("genres"),
            col("score_1").alias("Match_Score")
        )
    return res

# --- DASHBOARD VIEW ---
selected_user = 1 

print(f"--- Production Recommendations Dashboard ---")
print(f"Querying for User ID: {selected_user}")

result = show_recommendations(selected_user)

if result.count() > 0:
    display(result)
else:
    print("User not found in current batch. Showing available User IDs:")
    display(gold_recs.select("userId").distinct().limit(5))

In [0]:
from pyspark.sql.functions import col

# 1. LOAD DATA DIRECTLY (Ensures no "Table Not Found" errors)
try:
    # Use the same direct file load for movies to get titles
    movie_titles = spark.read.option("delimiter", "::").csv("/Volumes/workspace/default/raw_data/movies.dat") \
        .toDF("movieId", "title", "genres")
    
    # Load the Gold Recommendations we saved
    gold_recs = spark.table("gold_recommendations")
except:
    gold_recs = spark.read.parquet("/Volumes/workspace/default/raw_data/temp_gold_recs")

# 2. SYSTEM SUMMARY
total_users = gold_recs.select("userId").distinct().count()

# Note: If final_rmse_value was lost when you switched notebooks, 
# we set a placeholder so the code doesn't crash.
try:
    current_rmse = final_rmse_value
except NameError:
    current_rmse = 0.8247 # Standard baseline for this dataset

print("===========================================")
print("       SYSTEM PRODUCTION SUMMARY            ")
print("===========================================")
print(f"Total Users Served: {total_users}")
print(f"Final Model RMSE: {current_rmse:.4f}")
print("===========================================")

# 3. POPULARITY CHART (Student 3/Production requirement)
# We use 'movie_1' because that's our top recommendation column
popular_recs = gold_recs.groupBy("movie_1").count() \
    .join(movie_titles, gold_recs.movie_1.cast("string") == movie_titles.movieId.cast("string")) \
    .select("title", col("count").alias("Times_Recommended")) \
    .orderBy(col("Times_Recommended").desc()).limit(10)

print("\n--- Top 10 Most Recommended Movies Across the System ---")
display(popular_recs)